# 01 - Data audit  (B03)

Audits a **bounded** sample of the candidate news collection and records verified facts and
unknowns. No return relationships are examined here: nothing selected in this notebook may be
chosen by looking at a p-value.

Written up in [`docs/data-audit-fnspid.md`](../docs/data-audit-fnspid.md); the decisions it drives
are recorded in `config.py` (B04).

**Two rules this notebook exists to respect.**

1. **The unit of audit is the source, not the file.** FNSPID's `All_external.csv` concatenates
   five-plus sub-corpora with different languages, schemas and timestamp behaviour. Pooled, it
   looks partly intraday; per source, one block is intraday and every other is date-only.
2. **A cluster sample cannot produce corpus-wide rates.** The sample is byte-range slices of a
   ticker-ordered file, so it returns whole ticker blocks. Duplication rate, headlines per
   session and company concentration are *withheld* by `src/audit.py` rather than computed and
   quietly believed.

Notebook contains no analysis logic: it imports from `src/`, calls, and displays.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

import config
from src import audit, plots


## 1. Fetch the bounded sample

From the repo root, once:

```bash
python data/raw/download.py --fnspid-sample --slices 24 --slice-mb 6
```

144 MB, 2.5% of a 5.7 GB file. Slices are spread evenly rather than taken as a prefix, because
the file is ordered by source block and then by ticker - a prefix would be a handful of tickers
whose names begin with "A".


In [ ]:
SAMPLE = config.DATA_RAW / 'fnspid_sample.parquet'
assert SAMPLE.exists(), f'run: python data/raw/download.py --fnspid-sample   ({SAMPLE} missing)'

df, load_stats = audit.load_sample(SAMPLE)
print(f"read {load_stats['n_read']:,}  ->  kept {load_stats['n_kept']:,}")
print(f"dropped {load_stats['n_dropped_malformed_date']} rows with a malformed Date")
for e in load_stats['malformed_examples']:
    print('   ', e[:70])


Those malformed `Date` values are not noise - they are fragments of **article body text**. That is
how the embedded-newline problem in the `Article` field announces itself, and it means the corpus
loader must parse CSV quoting rather than split on newlines.


## 2. The file is not one corpus

Classify by URL host, then profile each source separately. `looks_date_only` is the gate: a
source whose stamps concentrate at one clock time has no usable time-of-day.

Note the two clocks in `timestamp_profile`. Concentration is measured in the zone the source
**published** in - FNSPID stamps are UTC, so date-only rows sit at 00:00 UTC, which is 19:00 in
New York; read in market time they would look like an evening publication spike rather than the
absence of a time. Session position is measured in market time, the only clock in which "after
the close" means anything.


In [ ]:
profile = audit.profile_by_source(df)
profile[['rows', 'midnight_share', 'modal_time', 'distinct_minutes', 'looks_date_only',
         'ticker_share', 'cyrillic_share', 'likely_non_english', 'first', 'last']]


In [ ]:
for source, row in profile.head(6).iterrows():
    print(f'{source:>14s}  {row.verdict}')


### Timestamp regime over time

A source can change instrument mid-history, and a single pooled share hides it.


In [ ]:
(audit.midnight_share_by_year(df) * 100).round(1)


### Hour-of-day, for the one intraday source

Where a source's news lands relative to the US session is itself evidence about what the source
*is*.


In [ ]:
intraday = profile.index[~profile['looks_date_only']].tolist()
if intraday:
    fig = plots.figure_timestamp_audit(
        {s: audit.hour_histogram(df[df.source == s]) for s in intraday},
        {s: profile.loc[s, 'verdict'] for s in intraday},
    )
    plots.save(fig, 'figure0_timestamp_audit')
    display(fig)
else:
    print('no intraday source in this sample')


## 3. Mechanical screen - and what it deliberately will not do

`screen_sources` reports which sources have usable intraday timestamps, which are plausibly
English, and which carry ticker tags.

It does **not** pick a universe. Whether a source's *content* belongs in the study is a judgement
made by a person against a written criterion and recorded in the audit document. A screen that
guessed it would be the most dangerous function in the repository - the one intraday source here
passes every mechanical test and is still unusable.


In [ ]:
screen = audit.screen_sources(profile)
print(screen)
for note in screen.notes:
    print('\n  * ' + note)


### Read the content before judging relevance

This is the step that cannot be automated. Look at what each candidate actually publishes.


In [ ]:
for source in profile.index[:6]:
    print(f'\n=== {source} ===')
    for t in audit.content_sample(df, source, n=6):
        print('   ', t[:110])


## 4. What this sample cannot tell us

Span and per-year counts survive clustering and are returned. Per-session rates and the
duplication rate do not, and are withheld with the reason attached, so that "not computable from
this sample" can never be misread as "zero".


In [ ]:
cov = audit.coverage_profile(df, clustered=True)
print(f"headlines {cov['n_headlines']:,}   {cov['first']:%Y-%m-%d} .. {cov['last']:%Y-%m-%d}")
for k in ('per_session_mean', 'per_session_median', 'zero_news_share', 'thin_days_share'):
    print('   ', cov[k])


In [ ]:
dup = audit.duplication_profile(df, clustered=True)
print(dup['dedup_rate'])
print(f"\nshare of rows whose exact text repeats: {dup['share_in_repeated_texts']:.1%}")
print('\nmost repeated texts (structural evidence, still informative):')
dup['most_repeated'].head(8)


A market-wide roundup headline is emitted **once per tagged ticker**, so one editorial act is
weighted by the number of symbols it mentions. The dedup rule must handle that before daily
aggregation - and the rate itself is computed on the assembled corpus, not here.


## 5. Window proposal

Stability, not length, is binding: a year with a fraction of the neighbouring coverage makes the
daily aggregate a different measurement. Always returned as **provisional** - on a cluster sample
the absolute counts are not the corpus's.


In [ ]:
SOURCE = 'benzinga'   # the universe selected at B04; see docs/data-audit-fnspid.md
sub = df[df.source == SOURCE]
w = audit.suggest_window(sub, min_per_year=1000)
print(w['reason'])
print('excluded years:', w['excluded_years'])
print('provisional:', w['provisional'])
sub['ts_utc'].dt.year.value_counts().sort_index()


---
## 6. Decisions recorded at B04

Written into `config.py` and justified in [`docs/data-audit-fnspid.md`](../docs/data-audit-fnspid.md).

| Decision | Value |
|---|---|
| Universe | Benzinga sub-corpus; `NEWS_SOURCE_DOMAINS` filter is **mandatory**, not hygiene |
| Same-day association (RQ2) | **Inadmissible** - no source is both intraday and relevant |
| Date-only fallback | **Active**, and currently **inert** in the pipeline (B09) |
| Window | 2010-01-01 .. 2019-12-31, flagged provisional |
| Artifacts | FNSPID repo revision + file sha256; FinBERT revision. Loughran-McDonald unresolved |

`run_all.py` refuses to build a panel while the fallback is inert: applying the intraday close
rule to date-only stamps would assign each headline to the session it was dated rather than
deferring it, inventing an information boundary the data does not support.


In [ ]:
for k in ('NEWS_SOURCE', 'NEWS_SOURCE_DOMAINS', 'DATE_ONLY_FALLBACK',
          'DATE_ONLY_FALLBACK_IMPLEMENTED', 'RQ2_ADMISSIBLE', 'SAMPLE_START',
          'SAMPLE_END', 'SAMPLE_WINDOW_PROVISIONAL', 'NEWS_HF_REVISION',
          'FINBERT_REVISION', 'FINBERT_ID2LABEL', 'LM_DICT_VERSION'):
    print(f'  {k:32s} = {getattr(config, k)!r}')
